# Study Assistant: Summarize Study Material and Generate Quiz Questions

This notebook builds a study assistant with LangChain. It reads study material from a PDF file or from text pasted directly into the notebook, condenses it into key points, and writes multiple choice questions with four options and a marked correct answer.

There is no vector database and no retrieval step. The document text is passed straight into the prompt, which keeps the setup light for studying a single document at a time.

Pipeline: load text, summarize into key points, generate the quiz from those key points.

## Install Dependencies

In [1]:
!pip install langchain==0.3.26
!pip install langchain-openai==0.3.28
!pip install PyPDF2==3.0.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.6 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.2
    Uninstalling packaging-26.2:
      Successfully uninstalled packaging-26.2
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.13
    Uninstalling langchain-1.3.13:
      Successfully uninstalled langchain-1.3.13
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph 1.2.9 requires langchain-core<2,>=1.4.7, but you have langchain-cor

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 36.0 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.45.0
    Uninstalling openai-2.45.0:
      Successfully uninstalled openai-2.45.0
^C


In [3]:
pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 16.1 MB/s eta 0:00:00


## Set the OpenAI API Key

The key is typed into a hidden prompt and stored in an environment variable. It never gets written into a notebook cell, so the saved file can be shared without leaking the credential.

In [1]:
from getpass import getpass
import os

os.environ['OPENAI_API_KEY'] = getpass('Enter Open AI API Key: ')

Enter Open AI API Key: ··········


## Load Study Material from a PDF

PyPDF2 opens the file and returns the text of each page. Page text usually comes back with line breaks in the middle of sentences because the extractor follows the layout of the page rather than the grammar. The whitespace is collapsed into single spaces so the model receives readable sentences.

In [4]:
import re
import PyPDF2


def load_pdf_text(pdf_path):
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        pages = [page.extract_text() or '' for page in reader.pages]

    text = ' '.join(pages)
    return re.sub(r'\s+', ' ', text).strip()

In [9]:
from google.colab import files

uploaded = files.upload()

Saving Prompt Engineering (1).pdf to Prompt Engineering (1) (1).pdf


In [10]:
# upload the PDF to the notebook environment first, then point to it here
PDF_PATH = 'Prompt Engineering (1).pdf'

study_material = load_pdf_text(PDF_PATH)

print('Characters extracted:', len(study_material))
print()
print(study_material[:600])

Characters extracted: 4923

What is Prompt Engineering? Prompt engineering is a practice within natural language processing (NLP) in artificial intelligence, where text is used to describe the task the AI should perform. Guided by this input, the AI generates an output, which could take various forms. The goal is to use human-understandable text to interact conversationally with models, allowing for flexibility in the model’ s performance due to the task description embedded in the prompt. What are Prompts? Prompts are detailed descriptions of the desired output from an AI model. They represent the interaction between th


## Connect to the Model

In [11]:
from langchain_openai import ChatOpenAI

chatgpt = ChatOpenAI(model='gpt-4o-mini', temperature=0)

## Summarization Chain

The prompt fixes the number of bullet points and the length of each one so the summary stays short enough to serve as the input for the quiz step. Temperature is set to 0 so repeated runs on the same document return close to the same key points.

In [12]:
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

SUMMARY_PROMPT = """You are a study assistant. Read the study material below and write the key
points a student needs to know.

Rules:
- Write exactly {num_points} bullet points
- One idea per bullet, no more than 25 words each
- Use only facts stated in the material, do not add outside information
- Start each bullet with a dash

Study material:
{study_material}
"""

summary_prompt = ChatPromptTemplate.from_template(SUMMARY_PROMPT)
summary_chain = summary_prompt | chatgpt | StrOutputParser()

In [13]:
summary = summary_chain.invoke({'study_material': study_material, 'num_points': 6})

print(summary)

- Prompt engineering is a practice in NLP where text describes the AI task, guiding the model's output generation.  
- Prompts are detailed descriptions that define user expectations and interactions with the AI model.  
- Effective prompt engineering relies on well-designed prompts that provide sufficient detail for the model.  
- Key components of a prompt include instruction, context, input data, and output indicator.  
- Techniques like role playing, clarity, and specification enhance prompt effectiveness and model responses.  
- Avoid pitfalls such as information overload, open-ended questions, and poor use of constraints when creating prompts.  


## Quiz Generation Chain

Free text output reads well but is hard to check. Asking the model for a schema returns Python objects instead, so the number of options and the position of the correct answer can be verified in code before anything is shown to the student.

In [14]:
from typing import List
from pydantic import BaseModel, Field


class QuizQuestion(BaseModel):
    question: str = Field(description='The question text')
    options: List[str] = Field(description='Exactly four answer options')
    answer_index: int = Field(description='Index of the correct option, 0 through 3')


class Quiz(BaseModel):
    questions: List[QuizQuestion] = Field(description='The generated quiz questions')


QUIZ_PROMPT = """You are a study assistant. Write {num_questions} multiple choice questions
that test the key points below.

Rules:
- Each question has exactly four options
- Exactly one option is correct
- The wrong options must be plausible but clearly wrong to a student who knows the key points
- Ask only about content covered in the key points, do not add outside information
- Vary which position holds the correct answer

Key points:
{summary}
"""

quiz_prompt = ChatPromptTemplate.from_template(QUIZ_PROMPT)
quiz_chain = quiz_prompt | chatgpt.with_structured_output(Quiz)

### Validate and Display

`validate_quiz` drops any question that does not have exactly four options or whose answer index falls outside that range. This guards the display step against a malformed response instead of trusting the model to always follow the rules in the prompt.

In [15]:
LABELS = ['a', 'b', 'c', 'd']


def validate_quiz(quiz):
    return [q for q in quiz.questions
            if len(q.options) == 4 and 0 <= q.answer_index < 4]


def show_quiz(questions):
    for number, question in enumerate(questions, start=1):
        print(f'{number}. {question.question}')
        for label, option in zip(LABELS, question.options):
            print(f'   {label}) {option}')
        print()

    print('Answers:')
    for number, question in enumerate(questions, start=1):
        label = LABELS[question.answer_index]
        print(f'{number}. {label}) {question.options[question.answer_index]}')

In [16]:
quiz = quiz_chain.invoke({'summary': summary, 'num_questions': 5})
questions = validate_quiz(quiz)

print(f'Kept {len(questions)} of {len(quiz.questions)} generated questions')
print()
show_quiz(questions)

Kept 5 of 5 generated questions

1. What is the primary purpose of prompt engineering in NLP?
   a) To create complex algorithms for AI
   b) To describe the AI task and guide output generation
   c) To analyze user data for better AI performance
   d) To develop new programming languages for AI

2. Which of the following is NOT a key component of a prompt?
   a) Instruction
   b) Context
   c) Output indicator
   d) User feedback

3. What technique can enhance the effectiveness of a prompt?
   a) Using vague language
   b) Role playing
   c) Avoiding context
   d) Limiting instructions

4. What should be avoided when creating prompts for AI models?
   a) Providing clear instructions
   b) Using specific constraints
   c) Information overload
   d) Defining user expectations

5. Effective prompt engineering relies on which of the following?
   a) Ambiguous prompts
   b) Well-designed prompts with sufficient detail
   c) Short and simple prompts
   d) Randomized input data

Answers:
1. 

## Run the Whole Assistant

Both chains wrapped in one function, so any block of study material can be turned into a summary and a quiz in a single call.

In [17]:
def study_assistant(text, num_points=6, num_questions=5):
    summary = summary_chain.invoke({'study_material': text, 'num_points': num_points})
    quiz = quiz_chain.invoke({'summary': summary, 'num_questions': num_questions})
    questions = validate_quiz(quiz)

    print('Summary:')
    print(summary)
    print()
    print('Quiz Questions:')
    print()
    show_quiz(questions)

## Example with Pasted Text

The same functions work on any content a student pastes in, not only on PDF files.

In [18]:
topic = 'Prompt Engineering for Agents'

pasted_material = """Prompt engineering involves designing and refining inputs to language models
to achieve desired outputs. In the context of agents, prompt engineering allows for better control
over how an agent interacts with the environment and solves specific tasks. This is particularly
useful in domains like robotics and conversational AI. By adjusting the structure and content of
the prompts, users can enhance an agent's performance on specific tasks."""

print(f'Topic: {topic}')
print()
study_assistant(pasted_material, num_points=3, num_questions=2)

Topic: Prompt Engineering for Agents

Summary:
- Prompt engineering is the process of designing inputs to language models for desired outputs.  
- It enables better control of agent interactions with the environment and task-solving.  
- Adjusting prompt structure and content can enhance an agent's performance in robotics and conversational AI.  

Quiz Questions:

1. What is prompt engineering primarily concerned with?
   a) Designing outputs for language models
   b) Creating inputs to language models for desired outputs
   c) Developing hardware for AI systems
   d) Improving user interfaces for software applications

2. How can adjusting prompt structure and content affect an agent's performance?
   a) It has no effect on performance
   b) It can enhance performance in robotics and conversational AI
   c) It only improves performance in text-based tasks
   d) It complicates the interaction with the environment

Answers:
1. b) Creating inputs to language models for desired outputs
2.

## Notes and Limits

- The whole document goes into the prompt, so a long book will exceed the context window. Split it and run the assistant on one section at a time.
- PyPDF2 reads the text layer of a PDF. A scanned or image based PDF returns empty strings and needs OCR before this notebook can use it.
- `validate_quiz` checks the shape of the response, not whether the marked answer is actually right. The answer key still comes from the model and should be read before a student relies on it.
- Quiz quality tracks summary quality. If a key point is vague, the questions built on it will be vague too.